# Fine-Tuner un LLM avec QLoRA

## Objectif

Apprendre à **fine-tuner un vrai LLM** sur ses propres données avec **QLoRA** — sans infrastructure coûteuse, juste un GPU T4 gratuit sur Google Colab.

## Prérequis
- Un compte Google (pour Google Colab)
- GPU T4 activé (`Exécution > Modifier le type d'exécution > T4 GPU`)
- Des notions de base en Python et Machine Learning

## Déroulé

| Étape | Thème | Durée |
|-------|-------|-------|
| 0 | Installation & Setup | 10 min |
| 1 | Vérification du GPU | 5 min |
| 2 | Chargement du modèle | 10 min |
| 3 | Tester le modèle de base | 15 min |
| 4 | Préparer le dataset | 20 min |
| 5 | Configurer QLoRA | 10 min |
| 6 | Lancer l'entraînement | 15 min |
| 7 | Comparer avant / après | 10 min |

> **💡 Ce workshop tourne sur Google Colab avec un GPU T4 gratuit.** Pas besoin d'infrastructure coûteuse — QLoRA permet de fine-tuner un LLM de 1B+ paramètres sur un seul GPU !

### 🛑 Information importante

À chaque fois que vous voyez `...` cela signifie que c'est à vous d'implémenter le code manquant pour compléter l'exercice. N'hésitez pas à demander de l'aide si vous êtes bloqué !

### Mais c'est quoi le fine-tuning exactement ?

Le fine-tuning consiste à prendre un modèle pré-entraîné et à le ré-entraîner sur des données spécifiques. C'est comme un consultant généraliste à qui on fait suivre une formation dans un secteur précis : il garde son bagage général mais acquiert une expertise métier. L'avantage : on repart d'un modèle déjà très capable, ce qui demande bien moins de données et de calcul qu'un entraînement depuis zéro.

### Et QLoRA là-dedans ?

Fine-tuner un LLM de plusieurs milliards de paramètres demande normalement des dizaines de GPU.  
QLoRA = **Q**uantized **Lo**w-**R**ank **A**daptation : au lieu de modifier tous les poids, on entraîne seulement **~1% des paramètres**, et on compresse le modèle en 4-bit.  
Résultat : on peut fine-tuner un modèle de **1B+ paramètres sur un seul GPU**.

---
## Étape 0 — Installation & Setup

### Sous-étape 0.1 — Installer les dépendances

On va utiliser **Unsloth** — une librairie qui optimise le fine-tuning de LLMs.  
Comparé à Hugging Face classique, Unsloth c'est :
- **2x plus rapide** à l'entraînement
- **60% moins de VRAM** consommée
- Compatible avec tous les modèles Llama, Mistral, Qwen...

> ⏳ Cette cellule prend ~2-3 minutes. Normale — profitez-en pour relire les concepts ci-dessus.

In [15]:
%pip install -q unsloth
%pip install -q datasets trl peft accelerate bitsandbytes

print("✅ Installation terminée !")

✅ Installation terminée !


---
## Étape 1 — Vérification du GPU

### Concept

Sans GPU, le fine-tuning d'un LLM prendrait des heures sur CPU. On vérifie que le matériel est bien disponible avant de charger quoi que ce soit.

Si CUDA n'est pas disponible, allez dans `Exécution > Modifier le type d'exécution > GPU T4`.

### Sous-étape 1.1 — Vérifier la disponibilité du GPU

In [16]:
import torch

print(f"PyTorch : {torch.__version__}")
print(f"CUDA disponible : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("❌ Pas de GPU ! Va dans Exécution > Modifier le type d'exécution > GPU T4")

PyTorch : 2.10.0+cu128
CUDA disponible : True
GPU : Tesla T4
VRAM : 15.6 GB


---
## Étape 2 — Chargement du modèle

### Concept

On va charger **Llama-3.2-1B-Instruct** via Unsloth. Le paramètre `load_in_4bit=True` active la **quantization NF4** : les poids passent de 32-bit à 4-bit, divisant par **8 la mémoire nécessaire**.

C'est le premier pilier de QLoRA — on charge un modèle compressé, et on n'entraîne que de petits adaptateurs LoRA par-dessus.

Documentation Unsloth : https://github.com/unslothai/unsloth

### Sous-étape 2.1 — Charger le modèle avec Unsloth

In [17]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 1024  # Longueur max des séquences
MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,             # None = détection auto (float16 sur T4)
    load_in_4bit = True,      # Activer la quantization 4-bit
)

print(f"✅ Modèle '{MODEL_NAME}' chargé !")
print(f"Paramètres totaux : {model.num_parameters():,}")

==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


✅ Modèle 'unsloth/Llama-3.2-1B-Instruct' chargé !
Paramètres totaux : 1,235,814,400


---
## Étape 3 — Tester le modèle de base

Avant de modifier quoi que ce soit, observons comment répond le modèle de base.  
On va noter ses réponses et les comparer avec celles d'après le fine-tuning.

### Sous-étape 3.1 — Appliquer le chat template

Les LLMs modernes utilisent un **chat template** : un format précis pour structurer les messages (system, user, assistant).  
C'est comme un protocole de communication que le modèle a appris pendant son entraînement.

In [18]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

print("✅ Chat template appliqué !")

✅ Chat template appliqué !


### Sous-étape 3.2 — Implémenter la fonction de génération

La fonction `generate_response()` doit :
1. Formater le prompt avec le chat template
2. Tokenizer l'entrée
3. Générer une réponse
4. Décoder et retourner uniquement la partie générée (pas le prompt)

Documentation `model.generate()` : https://huggingface.co/docs/transformers/main_classes/text_generation

In [19]:
def generate_response(prompt, system="Tu es un assistant technique concis.", max_new_tokens=200):
    """Génère une réponse du modèle pour un prompt donné."""

    # Mettre le modèle en mode inférence (désactive le dropout, plus rapide)
    FastLanguageModel.for_inference(model)

    messages = [
        {"role": "system",  "content": system},
        {"role": "user",    "content": prompt},
    ]

    # Tokenizer les messages avec apply_chat_template
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    # Générer la réponse
    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        do_sample=True,
    )

    # Décoder uniquement les nouveaux tokens (en utilisant la longueur de input_ids)
    prompt_length = inputs.input_ids.shape[1]
    response = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True)

    return response

In [20]:
# Test du modèle de base
test_prompt = "Explique-moi LoRA en 3 phrases."

print("🤖 Réponse du modèle DE BASE :")
print("-" * 50)
response_before = generate_response(test_prompt)
print(response_before)
print("-" * 50)
print("📝 Retiens cette réponse, on va la comparer après le fine-tuning !")

🤖 Réponse du modèle DE BASE :
--------------------------------------------------
LoRA (Long Range Communication) est un standard de communication à faible puissance, utilisé pour transmettre des données à longue distance pour des applications telles que les télécommunications, les réseaux de données et les services de communication.
--------------------------------------------------
📝 Retiens cette réponse, on va la comparer après le fine-tuning !


---
## Étape 4 — Préparer le dataset

### Concept

C'est **l'étape la plus importante** du fine-tuning.  
Comme on dit : *garbage in, garbage out*.

**Notre objectif** : apprendre au modèle à répondre en **français**, de façon **courte et pédagogique**, avec des **analogies concrètes** — le style parfait pour une UG tech.

#### Format des données

Pour le SFT (Supervised Fine-Tuning), on utilise des paires **instruction / réponse** :

```json
[
  {
    "instruction": "C'est quoi le fine-tuning ?",
    "response": "Le fine-tuning c'est..."
  }
]
```

> 💡 En production, il faut des **centaines à milliers** d'exemples bien écrits. Pour ce workshop, on utilise un petit dataset pour illustrer le principe — les effets seront visibles même avec peu d'exemples.

### Sous-étape 4.1 — Créer le fichier de données

Créez un fichier `data.json` avec **au moins 10 exemples** sur des concepts IA/ML vus en cours.  
Chaque exemple doit avoir :
- `instruction` : une question claire
- `response` : une réponse courte (~3-5 phrases), avec une analogie si possible

Quelques idées de sujets : LoRA, QLoRA, RLHF, fine-tuning vs RAG, distillation, overfitting...

On vous en donne 3 en exemple pour que vous voyiez le style attendu :

In [21]:
import json

data = [
    {
        "instruction": "C'est quoi le fine-tuning ?",
        "response": "Le fine-tuning, c'est prendre un modèle pré-entraîné et le réentraîner sur tes données spécifiques. Imagine un consultant généraliste à qui tu fais suivre une formation dans ton secteur : il garde son bagage général mais acquiert ton expertise métier. L'avantage clé : tu pars d'un modèle déjà très capable, ce qui demande beaucoup moins de données et de calcul qu'un entraînement depuis zéro."
    },
    {
        "instruction": "Quelle est la différence entre LoRA et le full fine-tuning ?",
        "response": "Le full fine-tuning modifie tous les poids du modèle : très performant, mais coûteux en GPU et risque d'oublier les connaissances générales (catastrophic forgetting). LoRA, lui, ajoute de petites matrices d'adaptation et ne touche qu'à ~1% des paramètres. C'est comme coller des post-its sur une encyclopédie plutôt que la réécrire entièrement. Résultat : 10x moins de VRAM, entraînement bien plus rapide, et le modèle garde sa culture générale."
    },
    {
        "instruction": "Pourquoi utiliser QLoRA plutôt que LoRA ?",
        "response": "QLoRA ajoute la quantization NF4 au LoRA classique : les poids du modèle sont compressés de 32-bit à 4-bit, ce qui divise la mémoire nécessaire par 8. Avec LoRA seul, il faut quand même charger le modèle complet en mémoire. Avec QLoRA, un modèle de 7B paramètres qui nécessitait 28 GB de VRAM peut tourner sur une simple RTX 4090 à 24 GB. C'est ce qui a démocratisé le fine-tuning pour les petites équipes."
    },
    {
        "instruction": "C'est quoi le RLHF ?",
        "response": "Le RLHF (Reinforcement Learning from Human Feedback) consiste à entraîner un modèle de récompense sur des préférences humaines, puis à l'utiliser pour affiner le LLM. C'est comme donner des notes à un élève : le modèle apprend à produire les réponses que les humains préfèrent. C'est cette technique qui a transformé GPT-3 en ChatGPT — elle aligne le modèle sur ce qu'on attend vraiment de lui."
    },
    {
        "instruction": "Quelle est la différence entre fine-tuning et RAG ?",
        "response": "Le fine-tuning modifie les poids du modèle pour lui apprendre un style ou des connaissances en profondeur — mais les données doivent être préparées à l'avance. Le RAG (Retrieval-Augmented Generation) lui fournit des documents à la volée au moment de la requête, sans modifier les poids. C'est la différence entre former un médecin (fine-tuning) et lui donner accès à une bibliothèque médicale pendant la consultation (RAG). Les deux sont complémentaires."
    },
    {
        "instruction": "C'est quoi l'overfitting ?",
        "response": "L'overfitting, c'est quand un modèle apprend trop bien les données d'entraînement au point de ne plus savoir généraliser. Imagine un élève qui mémorise les corrigés par cœur sans comprendre les concepts — il réussit les exercices connus, mais échoue sur des sujets nouveaux. En fine-tuning, on l'évite avec le dropout, un petit dataset varié, et en s'arrêtant au bon moment (early stopping)."
    },
    {
        "instruction": "C'est quoi un token ?",
        "response": "Un token est l'unité de base que traite un LLM : ce n'est ni un mot entier ni un caractère, mais quelque chose entre les deux. En pratique, 1 token ≈ ¾ de mot en anglais. 'Tokenizer' devient par exemple ['Token', 'izer']. Les LLMs ont une limite de contexte exprimée en tokens (ex: 128k tokens), et leur coût est souvent calculé au token. Comprendre les tokens aide à optimiser ses prompts."
    },
    {
        "instruction": "Comment fonctionne le mécanisme d'attention ?",
        "response": "L'attention permet à chaque token de 'regarder' les autres tokens de la séquence et de décider lesquels sont pertinents pour calculer sa représentation. C'est comme lire une phrase en soulignant les mots qui influencent le sens d'un mot particulier. Le mécanisme calcule des scores de similarité (Query × Key), les normalise (softmax), puis pondère les valeurs (Value). C'est le cœur de l'architecture Transformer."
    },
    {
        "instruction": "C'est quoi la distillation de modèle ?",
        "response": "La distillation consiste à entraîner un petit modèle (l'élève) à imiter les sorties d'un grand modèle (le professeur), pas juste les labels finaux mais les distributions de probabilité complètes. C'est comme transmettre le raisonnement d'un expert à un junior plutôt que juste ses conclusions. Résultat : un modèle compact qui retient une bonne partie des capacités du grand — parfait pour le déploiement en production."
    },
    {
        "instruction": "C'est quoi les données synthétiques ?",
        "response": "Les données synthétiques sont des données générées par un LLM pour entraîner un autre modèle (ou le même). Plutôt que de collecter et annoter des milliers d'exemples humains — coûteux et lent — on demande à GPT-4 de générer des conversations d'entraînement. C'est ce qu'Alpaca a fait avec GPT-3.5 pour entraîner LLaMA à moindre coût. Le risque : amplifier les biais ou erreurs du modèle générateur."
    },
]

# Sauvegarder le dataset
with open("data.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"✅ Dataset créé : {len(data)} exemples")
print(f"Premier exemple : {data[0]['instruction']}")

✅ Dataset créé : 10 exemples
Premier exemple : C'est quoi le fine-tuning ?


### Sous-étape 4.2 — Formater le dataset pour le chat template

Le modèle a été entraîné avec un format de conversation précis.  
On doit transformer nos paires `instruction / response` en conversations structurées avec les bons rôles : `system`, `user`, `assistant`.

Documentation `apply_chat_template` : https://huggingface.co/docs/transformers/chat_templating

In [22]:
SYSTEM_PROMPT = """Tu es un assistant pédagogique spécialisé en Intelligence Artificielle.
Tu réponds toujours en français, de façon concise (3-5 phrases max), avec des analogies concrètes."""

def format_to_chat(examples):
    """Convertit les paires instruction/response en conversations formatées."""
    conversations = []

    for instruction, response in zip(examples["instruction"], examples["response"]):
        messages = [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": instruction},
            {"role": "assistant", "content": response},
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

        conversations.append(text)

    return {"text": conversations}


# Charger le dataset depuis le JSON
from datasets import Dataset

with open("data.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)

# Convertir en Dataset HuggingFace
dataset = Dataset.from_list(raw_data)

# Appliquer le formatage
dataset = dataset.map(format_to_chat, batched=True)

print(f"✅ Dataset formaté : {len(dataset)} exemples")
print("\nExemple de texte formaté :")
print("-" * 50)
print(dataset[0]["text"][:300], "...")

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

✅ Dataset formaté : 10 exemples

Exemple de texte formaté :
--------------------------------------------------
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

Tu es un assistant pédagogique spécialisé en Intelligence Artificielle.
Tu réponds toujours en français, de façon concise (3-5 phrases max), avec des analogies concrètes.<|eot ...


---
## Étape 5 — Configurer QLoRA

### Concept clé : Comment LoRA fonctionne ?

LoRA ajoute deux petites matrices **A** et **B** à chaque couche d'attention.  
Au lieu de modifier les poids W directement, on apprend ΔW = B × A, où le rang `r` est très petit (4 à 64).  
Le nombre de paramètres entraînables passe de `d×d` à `2×d×r` — soit environ **1% du total**.

**Paramètres clés** :
- **`r`** : rang des matrices — plus grand = plus de paramètres = meilleur mais plus lent
- **`lora_alpha`** : facteur de scaling — contrôle l'intensité de la mise à jour des poids (généralement = `r` ou `2*r`)

Documentation PEFT/LoRA : https://huggingface.co/docs/peft/conceptual_guides/lora

### Sous-étape 5.1 — Appliquer les adaptateurs LoRA

In [23]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                 # Rang des matrices LoRA
                            # Plus r est grand → plus de paramètres → meilleur mais plus lent

    lora_alpha = 32,        # Facteur de scaling LoRA (= 2*r en général)
                            # Contrôle l'intensité de la mise à jour des poids

    lora_dropout = 0.05,    # Dropout pour éviter l'overfitting

    target_modules = [      # Quelles couches on fine-tune ?
        "q_proj",           # Query projection (attention)
        "k_proj",           # Key projection (attention)
        "v_proj",           # Value projection (attention)
        "o_proj",           # Output projection (attention)
        "gate_proj",        # FFN
        "up_proj",          # FFN
        "down_proj",        # FFN
    ],

    bias = "none",          # Pas de bias LoRA
    use_gradient_checkpointing = "unsloth",  # Économise encore de la VRAM
    random_state = 42,
)

# Afficher le ratio de paramètres entraînables
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())

print(f"Paramètres entraînables : {trainable:,} ({100 * trainable / total:.2f}% du total)")
print(f"Paramètres gelés        : {total - trainable:,}")
print("\n✅ QLoRA configuré ! On va entraîner seulement une infime partie du modèle.")

Paramètres entraînables : 11,272,192 (1.43% du total)
Paramètres gelés        : 774,440,960

✅ QLoRA configuré ! On va entraîner seulement une infime partie du modèle.


---
## Étape 6 — Lancer l'entraînement

### Concept

On utilise **SFTTrainer** (Supervised Fine-Tuning Trainer) de la librairie TRL de Hugging Face.  
C'est comme préparer un plan d'entraînement sportif : on définit combien d'époques (tours complets du dataset), à quelle intensité (learning rate), etc.

Documentation SFTTrainer : https://huggingface.co/docs/trl/sft_trainer

### Sous-étape 6.1 — Configurer les paramètres d'entraînement

In [24]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir = "./results",         # Dossier de sauvegarde

    num_train_epochs = 3,                # 3 passes complètes sur le dataset
    per_device_train_batch_size = 2,     # 2 exemples par step (T4 : 15 GB de VRAM)
    gradient_accumulation_steps = 4,     # Batch effectif = 2 × 4 = 8
                                         # Simule un batch plus grand sans plus de VRAM

    learning_rate = 2e-4,                # Learning rate pour LoRA (plus élevé qu'un full fine-tuning)
    lr_scheduler_type = "cosine",        # Décroissance cosinus du learning rate
    warmup_ratio = 0.1,                  # 10% des steps en warmup

    # Optimisations mémoire
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    optim = "adamw_8bit",               # Optimizer 8-bit pour économiser la VRAM

    # Logging & sauvegarde
    logging_steps = 1,
    save_strategy = "epoch",

    seed = 42,
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",       # La colonne créée à l'étape 4
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    args = training_args,
)

print("✅ Trainer prêt !")
print("\nRécap de la config :")
print(f"  Époques          : {training_args.num_train_epochs}")
print(f"  Batch size       : {training_args.per_device_train_batch_size}")
print(f"  Gradient accum   : {training_args.gradient_accumulation_steps}")
print(f"  Learning rate    : {training_args.learning_rate}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/10 [00:00<?, ? examples/s]

✅ Trainer prêt !

Récap de la config :
  Époques          : 3
  Batch size       : 2
  Gradient accum   : 4
  Learning rate    : 0.0002


In [25]:
# Afficher la VRAM utilisée avant l'entraînement
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU : {gpu_stats.name}")
print(f"VRAM réservée au départ : {start_gpu_memory} GB / {max_memory} GB")
print()

# Lancer l'entraînement
trainer_stats = trainer.train()

# Afficher la VRAM utilisée après
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"\n✅ Entraînement terminé !")
print(f"VRAM utilisée pendant l'entraînement : {used_memory} GB")
print(f"Temps total : {round(trainer_stats.metrics['train_runtime'] / 60, 2)} minutes")

GPU : Tesla T4
VRAM réservée au départ : 3.029 GB / 14.563 GB



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10 | Num Epochs = 3 | Total steps = 6
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss
1,3.630573
2,3.569248
3,3.018044
4,2.732517
5,2.466460
6,2.338789



✅ Entraînement terminé !
VRAM utilisée pendant l'entraînement : 3.863 GB
Temps total : 0.18 minutes


### Sous-étape 6.2 — Sauvegarder le modèle fine-tuné

Avec LoRA, on sauvegarde d'abord seulement les **adaptateurs** (très légers),  
puis on peut fusionner avec le modèle de base pour obtenir un modèle standalone.

Documentation : https://huggingface.co/docs/peft/tutorial/peft_model_config

In [26]:
SAVE_PATH = "./fine_tuned_model"

# Sauvegarder les adaptateurs LoRA
model.save_pretrained(SAVE_PATH)

# Sauvegarder le tokenizer
tokenizer.save_pretrained(SAVE_PATH)

print(f"✅ Modèle sauvegardé dans '{SAVE_PATH}'")
print("\nFichiers sauvegardés :")
import os
for f in os.listdir(SAVE_PATH):
    size = os.path.getsize(os.path.join(SAVE_PATH, f)) / 1e6
    print(f"  {f} ({size:.1f} MB)")

✅ Modèle sauvegardé dans './fine_tuned_model'

Fichiers sauvegardés :
  adapter_config.json (0.0 MB)
  tokenizer.json (17.2 MB)
  chat_template.jinja (0.0 MB)
  tokenizer_config.json (0.0 MB)
  README.md (0.0 MB)
  adapter_model.safetensors (45.1 MB)


---
## Étape 7 — Comparer avant / après

C'est le moment de vérité !  
On va poser les mêmes questions au modèle de base et au modèle fine-tuné, et comparer les réponses.

Si le fine-tuning a bien fonctionné, le modèle fine-tuné devrait répondre :
- ✅ En **français**
- ✅ De façon **concise** (3-5 phrases)
- ✅ Avec des **analogies concrètes**

In [27]:
# Charger le modèle fine-tuné depuis le disque
ft_model, ft_tokenizer = FastLanguageModel.from_pretrained(
    model_name = SAVE_PATH,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = True,
)
ft_tokenizer = get_chat_template(ft_tokenizer, chat_template="llama-3.1")

print("✅ Modèle fine-tuné rechargé !")

==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


✅ Modèle fine-tuné rechargé !


In [29]:
def generate_with(mdl, tok, prompt, system="Tu es un assistant technique concis.", max_new_tokens=200):
    """Génère une réponse avec un modèle et tokenizer donnés."""
    FastLanguageModel.for_inference(mdl)
    messages = [
        {"role": "system",  "content": system},
        {"role": "user",    "content": prompt},
    ]
    inputs = tok.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    outputs = mdl.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        do_sample=True
    )

    # Correction ici : on utilise inputs.input_ids.shape[1]
    prompt_length = inputs.input_ids.shape[1]
    return tok.decode(outputs[0][prompt_length:], skip_special_tokens=True)


# Questions de test : dans le dataset + hors dataset
test_questions = [
    "C'est quoi le fine-tuning ?",
    "Quelle est la différence entre LoRA et le full fine-tuning ?",
    "Pourquoi utiliser QLoRA plutôt que LoRA ?",
    "C'est quoi le catastrophic forgetting ?",   # Hors dataset — le modèle généralise-t-il ?
]

print("=" * 65)
for q in test_questions:
    print(f"\n❓ {q}\n")

    before = generate_with(model, tokenizer, q)
    after  = generate_with(ft_model, ft_tokenizer, q)

    print(f"🔵 AVANT  : {before}")
    print()
    print(f"🟢 APRÈS  : {after}")
    print("-" * 65)


❓ C'est quoi le fine-tuning ?

🔵 AVANT  : Le fine-tuning est un processus de rétroprojection de modèles d'apprentissage de grande taille (RNN) vers de petits modèles de grande taille (LSTM) pour financer des représentations finales plus robustes et plus adaptées aux données spécifiques.

🟢 APRÈS  : Le fine-tuning est un processus de modification des paramètres de prétraitement d'un signal audio pour obtenir une réponse spécifique. Il s'agit généralement d'ajuster les paramètres de voix, de fréquence, de dynamique, etc. pour parvenir à un résultat parfait. Le fine-tuning est souvent utilisé pour obtenir un son idéal en quelques minutes ou quelques heures.
-----------------------------------------------------------------

❓ Quelle est la différence entre LoRA et le full fine-tuning ?

🔵 AVANT  : LoRA (Large Model) et Full Fine-tuning sont deux techniques de traitement de langage automatique (TNA) qui ont été popularisés récemment.

**LoRA (Large Model)**

LoRA est une technique de trait

### Sous-étape 7.1 — Analyser les résultats

Observez les différences entre les deux modèles et répondez à ces questions :

1. **Le modèle répond-il bien en français ?** Avant / après ?
2. **Les réponses sont-elles plus concises** avec le modèle fine-tuné ?
3. **Y a-t-il des analogies** dans les réponses du modèle fine-tuné ?
4. **Pour les questions hors dataset** — le modèle généralise-t-il ou hallucine-t-il ?

> **💡 À retenir** : Avec seulement 10 exemples, on voit déjà un effet sur le style. Imaginez avec 500 exemples de qualité... C'est là que le fine-tuning devient vraiment puissant.

---
## Bonus — Pousser sur Hugging Face Hub

Vous voulez partager votre modèle fine-tuné ? Hugging Face Hub, c'est le GitHub des modèles IA.

**Prérequis** : créer un compte sur [huggingface.co](https://huggingface.co) et générer un token d'accès (Settings > Access Tokens).

In [ ]:
# Connexion à Hugging Face
from huggingface_hub import login

HF_TOKEN = "hf_..."  # Remplacez par votre token (https://huggingface.co/settings/tokens)
login(token=HF_TOKEN)

print("✅ Connecté à Hugging Face Hub !")

In [ ]:
HF_USERNAME = "votre-username"   # Remplacez par votre username HF
MODEL_REPO  = "llama-3.2-1B-pedagogique-fr"

repo_id = f"{HF_USERNAME}/{MODEL_REPO}"

# Fusionner les adaptateurs LoRA avec le modèle de base avant de pousser
# (produit un modèle standalone, plus facile à utiliser)
ft_model.push_to_hub_merged(
    repo_id,
    ft_tokenizer,
    save_method = "merged_16bit",   # Fusionne et sauvegarde en float16
    token = HF_TOKEN,
)

print(f"\n✅ Modèle poussé sur : https://huggingface.co/{repo_id}")
print("N'importe qui peut maintenant charger votre modèle avec :")
print(f'  from transformers import AutoModelForCausalLM')
print(f'  model = AutoModelForCausalLM.from_pretrained("{repo_id}")')

---
## Conclusion

### Ce qu'on a appris

1. **QLoRA rend le fine-tuning accessible** : quantization 4-bit + LoRA permettent de travailler sur un seul GPU T4
2. **Le dataset est roi** : la qualité des exemples détermine la qualité du résultat final
3. **Le chat template est crucial** : chaque modèle attend un format précis pour les conversations
4. **LoRA n'entraîne que ~1% des paramètres** : mais l'effet sur le style et le comportement est bien réel
5. **La comparaison avant/après** révèle concrètement l'impact du fine-tuning

### Prochaines étapes

- Augmenter la taille du dataset → les effets seront bien plus marqués
- Jouer avec les hyperparamètres (rang `r`, learning rate, époques)
- Essayer avec un modèle plus grand : `unsloth/Llama-3.2-3B-Instruct`
- Explorer le **RLHF** / **DPO** pour aligner le modèle sur des préférences humaines
- Combiner fine-tuning + **RAG** pour avoir à la fois le style ET les connaissances à jour

### Ressources

- [Documentation Unsloth](https://github.com/unslothai/unsloth)
- [Documentation PEFT/LoRA](https://huggingface.co/docs/peft/conceptual_guides/lora)
- [Documentation SFTTrainer (TRL)](https://huggingface.co/docs/trl/sft_trainer)
- [QLoRA : paper original](https://arxiv.org/abs/2305.14314)

Écrit avec grand amour par William Jolivet et Léandre Ramos
Rejoignez PoC et Taker <3